# Calibrated Speed Scaling — quick demo

A short tour of the simulator. Runs take about a minute; full thesis experiments are in `experiments/run_experiments.py`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
from css.engine import run_sim
from css.psi import psi_mg1, psi_mgn

## 1. The calibration factor ψ
The M/G/1 factor grows with load and job-size variability; the M/G/n factor stays close to 1 in a large cluster.

In [ ]:
for n, rho, cov in [(1, 0.7, 2.0), (10, 0.7, 2.0), (100, 0.7, 2.0), (100, 0.9, 2.0)]:
    print(f'n={n:>3} rho={rho} CoV={cov}:  psi_MG1={psi_mg1(rho, cov):6.3f}   psi_MGn={psi_mgn(n, rho, cov):6.3f}')

## 2. DVFS vs the full-speed baselines
Short run (50k jobs, one seed), so numbers differ slightly from the thesis benchmark.

In [ ]:
kw = dict(nt=50_000, warmup=5_000)
res = {p: run_sim(p, **kw) for p in ['simple', 'lwl', 'jsq', 'random']}
for p, r in res.items():
    print(f"{p:>7}: power={r['mean_power']:8,.0f} W   MRT={r['mean_resp']:7.1f} s   P95={r['p95_resp']:7.1f} s")
save = 1 - res['simple']['mean_power'] / res['lwl']['mean_power']
print(f'\nDVFS saving vs LWL: {save:.1%}')

## 3. The κ knob
κ weights delay against energy: larger κ saves more energy and costs more delay.

In [ ]:
kappas = [0.3, 0.5, 1.0, 2.0, 5.0]
pts = [run_sim('simple', kappa=k, **kw) for k in kappas]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([r['mean_resp'] for r in pts], [r['mean_power'] for r in pts], 'o-', color='steelblue', label='DVFS')
for k, r in zip(kappas, pts):
    ax.annotate(f'κ={k:g}', (r['mean_resp'], r['mean_power']), xytext=(6, 4), textcoords='offset points')
ax.plot(res['lwl']['mean_resp'], res['lwl']['mean_power'], '^', color='green', ms=12, label='LWL')
ax.set_xlabel('Mean response time (s)'); ax.set_ylabel('Power (W)'); ax.legend(); ax.grid(ls='--', alpha=0.5)
plt.show()

## 4. Thesis figures
All thesis figures are in `results/figures/` and can be rebuilt from the saved results with `python experiments/make_figures.py`.